# 00 Colab bootstrap

Clones the repo, installs the locked environment, pulls DVC data and points MLflow at DagsHub.
Add `MLFLOW_TRACKING_USERNAME`, `MLFLOW_TRACKING_PASSWORD` and `MLFLOW_TRACKING_URI` under Colab **Secrets** (key icon) and enable notebook access. Never paste tokens into a cell.

In [ ]:
import os
import subprocess

from google.colab import userdata

REPO_URL = "https://github.com/webmaster99-stack/credit-card-fraud-detection.git"
REPO_DIR = "/content/fraud-detection"
BRANCH = "main"


def sh(cmd, cwd=None, secret=False):
    """Run a shell command. With secret=True a failure hides the command (it may hold a token)."""
    try:
        subprocess.run(cmd, shell=True, check=True, cwd=cwd)
    except subprocess.CalledProcessError as e:
        if not secret:
            raise
        raise RuntimeError(f"Command failed (exit {e.returncode}); command hidden") from None


if not os.path.isdir(REPO_DIR):
    sh(f"git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}")
else:
    sh("git pull --ff-only", cwd=REPO_DIR)

In [ ]:
sh("pip install -q uv")
sh("uv sync --locked", cwd=REPO_DIR)

In [ ]:
# Credentials from Colab Secrets -> environment (also used by DVC's DagsHub remote).
# The DVC remote is assumed to be named "origin"; adjust once the remote is configured.
for key in ("MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
    os.environ[key] = userdata.get(key)

user, token = os.environ["MLFLOW_TRACKING_USERNAME"], os.environ["MLFLOW_TRACKING_PASSWORD"]
sh("uv run dvc remote modify --local origin auth basic", cwd=REPO_DIR)
sh(f"uv run dvc remote modify --local origin user {user}", cwd=REPO_DIR, secret=True)
sh(f"uv run dvc remote modify --local origin password {token}", cwd=REPO_DIR, secret=True)
sh("uv run dvc pull", cwd=REPO_DIR)

In [ ]:
# Sanity check: a trivial run should appear in DagsHub MLflow with its lineage tags.
# Requires a clean git tree (Colab clone is clean unless you edited files).
sh("uv run python -m fraud.models.smoke_run", cwd=REPO_DIR)